<div dir="rtl" align="right">

# تحويلُ هيلبرتَ وغلافُ السعةِ \(Hilbert Transform\)

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

يَستخرجُ تحويلُ هيلبرتَ الغلافَ اللحظيَّ للسعةِ من الإشارة. نَبني إشارةً مُركّبةً (analytic signal) من الإشارةِ الحقيقيّة، والقيمةُ المطلقةُ منها تُعطي غلافَ السعةِ اللحظيّ.

## المُخرجاتُ المُتوقّعةُ

- غلافٌ أحمرٌ يَتتبّعُ قممَ الإشارةِ من الجانبينِ
- المناطقُ ذاتُ الغلافِ العالي تَدلُّ على فتراتِ نشاطٍ مكثّفٍ
- المناطقُ ذاتُ الغلافِ المنخفضِ تَدلُّ على فتراتِ هدوءٍ

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | المعنى |
| --- | --- | --- |
| القناةُ | P4 | المنطقةُ الجداريةُ |
| معدّلُ الأخذِ | 200 Hz | عيّنةٌ كلَّ 5 ms |
| العيّناتُ المرسومةُ | 5000 | أولُ 25 ثانيةً |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly wfdb pywt


<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

نَنزّلُ مُشاركًا واحدًا فقط (`--subjects 1`) لتسريعِ التجربةِ في بيئةِ Colab.

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2، قناةَ P4 (المنطقةُ الجداريةُ).

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')


<div dir="rtl" align="right">

## 4. تطبيقُ تحويلِ هيلبرتَ

نَستخدمُ `scipy.signal.hilbert` لِبناءِ الإشارةِ التحليليّةِ المُركّبة، ثمّ نَستخرجُ القيمةَ المطلقةَ لِحسابِ غلافِ السعةِ اللحظيّ.

نُطبّقُ دالةَ clean_signal لِتنظيفِ الإشارةِ من الضوضاءِ بِترشيحِ نطاقٍ $1$-$45\,\text{Hz}$ وتَرشيحٍ شَقّيٍّ $50\,\text{Hz}$.

</div>

In [ ]:
from scipy.signal import hilbert, butter, filtfilt, iirnotch

def clean_signal(signal, fs=200, low=1.0, high=45.0, notch_freq=50.0):
    b_bp, a_bp = butter(4, [low / (fs / 2), high / (fs / 2)], btype='band')
    filtered = filtfilt(b_bp, a_bp, signal)
    b_notch, a_notch = iirnotch(notch_freq, 30.0, fs=fs)
    filtered = filtfilt(b_notch, a_notch, filtered)
    return filtered

channel_data = clean_signal(channel_data, fs=fs)

n_plot = min(5000, len(channel_data))
signal = channel_data[:n_plot]

analytic_signal = hilbert(signal)
amplitude_envelope = np.abs(analytic_signal)
print(f'Envelope range: {amplitude_envelope.min():.2f} - {amplitude_envelope.max():.2f} uV')

<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- الغلافُ الأحمرُ يُحيطُ بالإشارةِ من الجانبينِ
- المناطقُ ذاتُ الغلافِ العالي تَدلُّ على نشاطٍ مكثّفٍ
- استخدمْ أداةَ التكبيرِ لِفحصِ نطاقاتٍ زمنيةٍ مُحدّدةٍ


</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

t_sec = np.arange(n_plot) / fs

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Original signal - Channel P4',
                                    'Signal with amplitude envelope - Channel P4'))
fig.add_trace(go.Scatter(x=t_sec, y=signal, name='Signal',
                         line=dict(color='blue', width=0.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=signal, name='Signal',
                         line=dict(color='blue', width=0.5, opacity=0.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=amplitude_envelope, name='Envelope',
                         line=dict(color='red', width=1.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=-amplitude_envelope, name='-Envelope',
                         line=dict(color='red', width=1.5)), row=2, col=1)
fig.update_layout(height=700, title_text='Hilbert Transform - Amplitude Envelope - Channel P4',
                  xaxis_title='Time (s)', xaxis2_title='Time (s)',
                  yaxis_title='Amplitude (uV)', yaxis2_title='Amplitude (uV)',
                  showlegend=False)
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- تحويلُ هيلبرتَ يَستخرجُ الغلافَ اللحظيَّ للسعةِ من الإشارةِ
- الغلافُ يَتتبّعُ قممَ الإشارةِ ويُلخّصُ طاقتَها اللحظيّةَ
- يُستخدمُ في قياسِ قوّةِ النشاطِ الدماغيِّ في نطاقٍ تردديٍّ مُحدّدٍ عبرَ الزمن
- يُستخدمُ أيضاً في دراساتِ ترابطِ الدماغِ لقياسِ التزامنِ بينَ المناطقِ


</div>